# MCP OAuth：Audience、Scope 与资源级授权

**面试问题：Agent 调 MCP 工具时，为什么一个签名有效的 Access Token 仍可能不能用？**

## 回答主线

1. OAuth Access Token 的签名只证明由可信授权服务器签发，不代表它适用于当前 MCP Server。
2. Resource Server 必须验证 issuer、audience、expiry、scope 和主体，而不是只解码 claims。
3. 工具级 scope 仍不够，订单、租户等资源还要做对象级授权。
4. Agent 不应把上游宽权限 Token 原样转发给下游工具，应通过 Token Exchange 获得目标 audience 的最小 scope Token。
5. 错误要区分 unauthenticated、wrong audience 和 insufficient scope，方便安全恢复。
6. 生产还需要标准 JWT/JWK、轮换、撤销和 DPoP/mTLS。

## 真实案例

一个订单 MCP Server 接收六枚教学 Token：合法读、错误 audience、过期、缺退款 scope、其他用户订单和管理员退款。我们用 HMAC 构造可验证 Token，先复现只验签名的 confused deputy，再实现完整授权矩阵和下游降权 Token Exchange。案例使用仓库内生成的离线脱敏小数据，输出用于学习机制，不代表生产性能。

### 输入预览：六个 Token Claim 与工具请求

In [1]:
import base64  # 导入 URL-safe Base64 以编码教学 Token。
import hashlib  # 导入 SHA-256 供 HMAC 使用。
import hmac  # 导入消息认证码以验证教学签名。
import json  # 导入 JSON 以序列化 Claim。

DEMO_KEY = b"mcp-teaching-key"  # 定义仅供离线教学的共享密钥。
now = 1000  # 固定逻辑时间保证输出可复现。
cases = [  # 构造六个访问令牌与资源请求。
    {"id": "C1", "claims": {"iss": "auth.example", "sub": "user-1", "aud": "orders-mcp", "exp": 2000, "scope": ["orders:read"]}, "tool": "order.read", "order": "O1"},  # 合法读取自己的订单。
    {"id": "C2", "claims": {"iss": "auth.example", "sub": "user-1", "aud": "billing-api", "exp": 2000, "scope": ["orders:read"]}, "tool": "order.read", "order": "O1"},  # Token audience 指向其他服务。
    {"id": "C3", "claims": {"iss": "auth.example", "sub": "user-1", "aud": "orders-mcp", "exp": 900, "scope": ["orders:read"]}, "tool": "order.read", "order": "O1"},  # 已过期 Token。
    {"id": "C4", "claims": {"iss": "auth.example", "sub": "user-1", "aud": "orders-mcp", "exp": 2000, "scope": ["orders:read"]}, "tool": "refund.create", "order": "O1"},  # 缺少写 scope。
    {"id": "C5", "claims": {"iss": "auth.example", "sub": "user-1", "aud": "orders-mcp", "exp": 2000, "scope": ["orders:read"]}, "tool": "order.read", "order": "O2"},  # 试图读其他主体订单。
    {"id": "C6", "claims": {"iss": "auth.example", "sub": "admin-1", "aud": "orders-mcp", "exp": 2000, "scope": ["orders:read", "refund:write", "orders:admin"]}, "tool": "refund.create", "order": "O2"},  # 管理员合法退款。
]  # 完成授权矩阵。
order_owners = {"O1": "user-1", "O2": "user-2"}  # 定义对象级订单归属。
print("案例  sub      aud          exp   scope                               tool/order")  # 输出输入表头。
for case in cases:  # 逐案例展示 Claim 和目标资源。
    claims = case["claims"]  # 读取当前 Token Claim。
    print(f"{case['id']}   {claims['sub']:<8} {claims['aud']:<12} {claims['exp']:>4}  {str(claims['scope']):<35} {case['tool']}/{case['order']}")  # 展示六类边界。

案例  sub      aud          exp   scope                               tool/order
C1   user-1   orders-mcp   2000  ['orders:read']                     order.read/O1
C2   user-1   billing-api  2000  ['orders:read']                     order.read/O1
C3   user-1   orders-mcp    900  ['orders:read']                     order.read/O1
C4   user-1   orders-mcp   2000  ['orders:read']                     refund.create/O1
C5   user-1   orders-mcp   2000  ['orders:read']                     order.read/O2
C6   admin-1  orders-mcp   2000  ['orders:read', 'refund:write', 'orders:admin'] refund.create/O2


## Baseline 基线：只验证 Token 签名

In [2]:
def b64url(data):  # 把字节编码为无填充 URL-safe Base64。
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")  # 返回 JWT 风格字符串。

def sign_claims(claims):  # 构造教学版 header.payload.signature Token。
    header = b64url(json.dumps({"alg": "HS256", "typ": "JWT"}, sort_keys=True).encode("utf-8"))  # 编码固定 Header。
    payload = b64url(json.dumps(claims, sort_keys=True).encode("utf-8"))  # 编码 Claim Payload。
    signature = b64url(hmac.new(DEMO_KEY, f"{header}.{payload}".encode("ascii"), hashlib.sha256).digest())  # 计算 HMAC 签名。
    return f"{header}.{payload}.{signature}"  # 拼接教学 Token。

def decode_and_verify_signature(token):  # 只验证签名并解码 Payload。
    header, payload, signature = token.split(".")  # 拆分三个组成部分。
    expected = b64url(hmac.new(DEMO_KEY, f"{header}.{payload}".encode("ascii"), hashlib.sha256).digest())  # 重算签名。
    if not hmac.compare_digest(signature, expected):  # 使用常量时间比较签名。
        return None  # 签名错误时拒绝。
    padding = "=" * (-len(payload) % 4)  # 恢复 Base64 填充。
    return json.loads(base64.urlsafe_b64decode(payload + padding))  # 解码并返回 Claim。

tokens = [sign_claims(case["claims"]) for case in cases]  # 为六组 Claim 签发教学 Token。
baseline_decisions = [decode_and_verify_signature(token) is not None for token in tokens]  # 错误地把验签成功等同授权成功。
print("案例  签名有效  Baseline执行")  # 输出基线结果表头。
for case, decision in zip(cases, baseline_decisions):  # 逐案例展示全部被放行。
    print(f"{case['id']}    True      {decision}")  # 暴露 wrong-aud、过期和越权资源均穿透。

案例  签名有效  Baseline执行
C1    True      True
C2    True      True
C3    True      True
C4    True      True
C5    True      True
C6    True      True


### 核心实现：Claim、Scope 与对象级授权

In [3]:
required_scope = {"order.read": "orders:read", "refund.create": "refund:write"}  # 映射工具到最小权限。
def authorize(token, tool, order_id):  # 对单次 MCP 工具调用执行完整授权。
    claims = decode_and_verify_signature(token)  # 首先验证签名并解析 Claim。
    if claims is None:  # 签名无效属于认证失败。
        return {"allowed": False, "reason": "invalid-signature"}  # 返回明确错误。
    if claims.get("iss") != "auth.example":  # 检查可信授权服务器。
        return {"allowed": False, "reason": "wrong-issuer"}  # 拒绝其他发行者。
    if claims.get("aud") != "orders-mcp":  # 检查 Token 是否明确发给当前资源服务器。
        return {"allowed": False, "reason": "wrong-audience"}  # 阻止 confused deputy。
    if claims.get("exp", 0) <= now:  # 检查固定逻辑时间下是否过期。
        return {"allowed": False, "reason": "expired"}  # 拒绝过期 Token。
    if required_scope[tool] not in claims.get("scope", []):  # 检查当前工具最小 scope。
        return {"allowed": False, "reason": "insufficient-scope"}  # 拒绝 scope 不足。
    owner = order_owners[order_id]  # 读取目标订单权威主体。
    is_admin = "orders:admin" in claims.get("scope", [])  # 判断 Token 是否具备跨主体管理权限。
    if claims.get("sub") != owner and not is_admin:  # 普通用户只能访问自己的订单。
        return {"allowed": False, "reason": "resource-owner-mismatch"}  # 拒绝对象级越权。
    return {"allowed": True, "reason": "authorized", "principal": claims["sub"], "aud": claims["aud"]}  # 返回授权主体和 audience。

authorization_rows = [authorize(token, case["tool"], case["order"]) for token, case in zip(tokens, cases)]  # 运行六案例授权矩阵。
print("案例  allowed  reason")  # 输出结构化授权结果表头。
for case, result in zip(cases, authorization_rows):  # 逐案例展示失败层次。
    print(f"{case['id']}    {str(result['allowed']):<7} {result['reason']}")  # 展示只有 C1 和 C6 通过。

案例  allowed  reason
C1    True    authorized
C2    False   wrong-audience
C3    False   expired
C4    False   insufficient-scope
C5    False   resource-owner-mismatch
C6    True    authorized


## 结果解读：认证成功不等于资源授权成功

In [4]:
expected = [True, False, False, False, False, True]  # 定义六个案例的真实授权期望。
baseline_correct = sum(actual == target for actual, target in zip(baseline_decisions, expected)) / len(expected)  # 计算只验签名正确率。
strict_correct = sum(result["allowed"] == target for result, target in zip(authorization_rows, expected)) / len(expected)  # 计算完整授权正确率。
print("案例  验签放行  完整授权  解释")  # 输出同口径对照表头。
for case, baseline, result in zip(cases, baseline_decisions, authorization_rows):  # 对齐两种结论。
    print(f"{case['id']}    {str(baseline):<8} {str(result['allowed']):<8} {result['reason']}")  # 展示每个拒绝层。
print(f"授权判定正确率 {baseline_correct:.1%} -> {strict_correct:.1%}")  # 量化完整验证价值。
print("解读：C2 虽由可信服务器签名，却发给 billing-api；orders-mcp 接受它就成为 confused deputy。")  # 解释 audience 的核心作用。

案例  验签放行  完整授权  解释
C1    True     True     authorized
C2    True     False    wrong-audience
C3    True     False    expired
C4    True     False    insufficient-scope
C5    True     False    resource-owner-mismatch
C6    True     True     authorized
授权判定正确率 33.3% -> 100.0%
解读：C2 虽由可信服务器签名，却发给 billing-api；orders-mcp 接受它就成为 confused deputy。


## 失败案例：Agent 把上游宽 Token 原样转给下游

In [5]:
upstream_claims = {"iss": "auth.example", "sub": "admin-1", "aud": "agent-gateway", "exp": 2000, "scope": ["orders:read", "refund:write", "users:delete", "orders:admin"]}  # 构造上游宽权限 Token。
upstream_token = sign_claims(upstream_claims)  # 签发只面向 Agent Gateway 的 Token。
forward_result = authorize(upstream_token, "order.read", "O2")  # 错误地直接转发给 orders-mcp。
exchanged_claims = {"iss": "auth.example", "sub": "admin-1", "aud": "orders-mcp", "exp": 1100, "scope": ["orders:read", "orders:admin"], "act": "agent-gateway"}  # 模拟 Token Exchange 降权且缩短有效期。
exchanged_token = sign_claims(exchanged_claims)  # 签发目标 audience 的最小权限 Token。
exchange_result = authorize(exchanged_token, "order.read", "O2")  # 使用降权 Token 调目标工具。
delete_scope_forwarded = "users:delete" in upstream_claims["scope"]  # 检查原 Token 携带无关高危权限。
delete_scope_exchanged = "users:delete" in exchanged_claims["scope"]  # 检查下游 Token 已移除无关权限。
print(f"原样转发：{forward_result}，含users:delete={delete_scope_forwarded}")  # 展示 wrong audience 被拒绝。
print(f"Token Exchange：{exchange_result}，含users:delete={delete_scope_exchanged}")  # 展示目标化最小权限。
print("修正策略：每一跳用 RFC Token Exchange 或等价机制获得 audience 限定、短时、最小 scope Token，并保留 act 委托链。")  # 总结委托边界。

原样转发：{'allowed': False, 'reason': 'wrong-audience'}，含users:delete=True
Token Exchange：{'allowed': True, 'reason': 'authorized', 'principal': 'admin-1', 'aud': 'orders-mcp'}，含users:delete=False
修正策略：每一跳用 RFC Token Exchange 或等价机制获得 audience 限定、短时、最小 scope Token，并保留 act 委托链。


### 生产边界与授权事件

In [6]:
auth_event = {"request_id": "C2", "subject": "user-1", "tool": "order.read", "resource": "O1", "token_aud": "billing-api", "server_aud": "orders-mcp", "decision": "deny", "reason": "wrong-audience"}  # 构造无敏感 Token 内容的审计事件。
print("授权事件：", auth_event)  # 展示排查字段而不记录原始 Access Token。
print("生产替换点：真实系统需要标准 OAuth AS metadata、JWK 轮换、JWT/opaque introspection、资源指示器、撤销、DPoP/mTLS 和租户 ACL。")  # 明确教学 HMAC Token 边界。

授权事件： {'request_id': 'C2', 'subject': 'user-1', 'tool': 'order.read', 'resource': 'O1', 'token_aud': 'billing-api', 'server_aud': 'orders-mcp', 'decision': 'deny', 'reason': 'wrong-audience'}
生产替换点：真实系统需要标准 OAuth AS metadata、JWK 轮换、JWT/opaque introspection、资源指示器、撤销、DPoP/mTLS 和租户 ACL。


## 回归测试：最后只保护 Audience、Scope、资源与降权

In [7]:
assert authorization_rows[0]["allowed"] and authorization_rows[5]["allowed"]  # 验证普通用户自有读取和管理员退款通过。
assert authorization_rows[1]["reason"] == "wrong-audience" and authorization_rows[2]["reason"] == "expired"  # 验证 audience 与过期错误分层。
assert authorization_rows[3]["reason"] == "insufficient-scope" and authorization_rows[4]["reason"] == "resource-owner-mismatch"  # 验证工具 scope 和对象级 ACL。
assert strict_correct == 1.0 and strict_correct > baseline_correct  # 验证完整授权矩阵符合全部期望。
assert not forward_result["allowed"] and exchange_result["allowed"] and delete_scope_forwarded and not delete_scope_exchanged  # 验证原样转发失败而降权交换成功。
print("回归测试通过：合法访问、Audience、过期、Scope、对象 ACL 和 Token Exchange 均成立。")  # 用少量断言总结 MCP OAuth 合同。

回归测试通过：合法访问、Audience、过期、Scope、对象 ACL 和 Token Exchange 均成立。
